## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install datasets transformers pandas

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoImageProcessor, ResNetForImageClassification, AutoModelForImageClassification 
from datasets import load_dataset, load_from_disk, concatenate_datasets
import copy
import torch.nn.functional as F
import os
from collections import defaultdict

## Dataset Prep

In [ ]:
processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")

In [ ]:
train = load_from_disk("/workspace/preprocessed/Cifar-10/train_converted2")
val = load_from_disk("/workspace/preprocessed/Cifar-10/val_converted2")
test = load_from_disk("/workspace/preprocessed/Cifar-10/test_converted2")

In [ ]:
train.set_format(type="torch", columns=["label", "img", "pixel_values"])
val.set_format(type="torch", columns=["label", "img", "pixel_values"])
test.set_format(type="torch", columns=["label", "img", "pixel_values"])

def clip_collate_fn(batch):
    # images = np.stack([example["pixel_values"] for example in batch])
    # images = torch.from_numpy(images)
    # labels = torch.tensor([example["label"] for example in batch])
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
val_loader = DataLoader(val, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)
test_loader = DataLoader(test, batch_size=64, shuffle=False, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

## Model Prep

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def leastSquares(Z0, Z1):
    W, residuals, rank, s = np.linalg.lstsq(Z0, Z1, rcond=None)
    return W, residuals

def cosineSimilarity(fine, aug):
    # Prevent NaNs
    eps = 1e-8
    out_aug = F.normalize(aug, dim=1, eps=eps)
    out_fine = F.normalize(fine, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

In [ ]:
# ResNet50 HF GitHub: https://github.com/huggingface/transformers/blob/main/src/transformers/models/resnet/modeling_resnet.py
# Uses Global Average Pooling (GAP) instead of CLS Tokens like ViT. GAP is only applied at the end, whereas ViT is always present. 
# Look at lines #315 -> #270 -> #206 in the GitHub
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.GAP = []
    
    def forward(self, pixel_values):
        self.GAP = []
        x = self.model.resnet.embedder(pixel_values)
        for i, stage_module in enumerate(self.model.resnet.encoder.stages): # Using ResNetBottleNeck blocks for ResNet50. 4 of them total. {0,1,2,3}
            if i == 3:
                for layer in stage_module.layers:
                    x = layer(x) # (Batch_Size, 2048, H, W); Return output of each block after all convolutions and projection if needed
                    pooled_output = self.model.resnet.pooler(x).squeeze(-1).squeeze(-1)
                    self.GAP.append(pooled_output) # (Batch_Size, 2048)
            else:
                x = stage_module(x) # Full forward pass of all stages
        
        return self.GAP # List of 3 tensors

In [ ]:
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(in_features=2048, out_features=10)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage

    def forward(self, pixel_values):
        x = self.model.resnet.embedder(pixel_values)
        for i, stage_module in enumerate(self.model.resnet.encoder.stages):
            if i == 3:
                for j, layer in enumerate(stage_module.layers):
                    x = layer(x)
                    if j == self.transform_stage:
                        break
            else:
                x = stage_module(x)
        
        pooled_output = self.model.resnet.pooler(x).squeeze(-1).squeeze(-1) # Shape: (batch_size, hidden_dim); Used for final comparison
        if self.W is not None:
            pooled_output = pooled_output @ self.W 

        logits = self.classifier(pooled_output) # Shape: (batch_size, num_classes)
        return logits, pooled_output # Return Predictions and GAP

In [ ]:
refer = ResNetForImageClassification.from_pretrained("microsoft/resnet-50").to(device)
base_H = Hooks(copy.deepcopy(refer)).to(device)
fine_tuned_H = Hooks(AutoModelForImageClassification.from_pretrained("yhyan/resnet-50-finetuned-eurosat")).to(device) # https://huggingface.co/yhyan/resnet-50-finetuned-eurosat

In [ ]:
base = Augmented(copy.deepcopy(refer)).to(device)
f_t = AutoModelForImageClassification.from_pretrained("yhyan/resnet-50-finetuned-eurosat")
fine_tuned = Augmented(copy.deepcopy(f_t), classifier=f_t.classifier).to(device)

In [ ]:
size_nums = [i for i in range(10, 101, 10)] #10
size_nums.insert(0, 1) # 11
size_nums.insert(1, 5) # 12
size_nums.extend([150, 200, 225, 250, 1000, 2500, float('inf')]) # 19 range(19)
train_size = 0

labels = train["label"]

label_to_indices = defaultdict(list)

for idx, label in enumerate(labels):
    label = int(label)
    label_to_indices[label].append(idx)

In [ ]:
filtered_train = {
    label: train.select(indices) for label, indices in label_to_indices.items()
}

for label, ds in filtered_train.items():
    print(f"Label {label}: {len(ds)} examples")

In [ ]:
indices = [i for i in range(0, 3)] # 3 total. For the last 3 layer outputs of stage 3. 

for s in range(len(size_nums)):
    train_indice = s

    label_counts = defaultdict(int)
    new_examples = []

    for i in range(10): # 10 labels
        ds = filtered_train[i]
        num = min(size_nums[train_indice], len(ds))
        # For random sampling of images if needed
        # import random
        # indices = random.sample(range(len(ds)), num)
        # ds = ds.select(indices)
        ds = ds.select(range(num))
        new_examples.append(ds)
        

    train_dataset = concatenate_datasets(new_examples)
    train_size = len(train_dataset)

    train_dataset.set_format(type="torch", columns=["label", "img", "pixel_values"])
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=clip_collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

    print(f"Results for {size_nums[train_indice]} Image(s) per label: {train_size} Training Images -> 10000 Test Images")
    
    # Extracting Embeddings
    Z0 = {i: [] for i in indices}

    Z1_last_lsr = []

    with torch.no_grad():
        for batch in tqdm(train_loader, desc=f"Extracting Train Dataset Vectors"):
            images = batch["pixel_values"].to(device, non_blocking=True)

            out_base = base_H(images)
            out_fine_tuned = fine_tuned_H(images)
            
            for key in indices:
                Z0[key].append(out_base[key].float().cpu())

            Z1_last_lsr.append(out_fine_tuned[indices[-1]].float().cpu())

    Z1_last_lsr = torch.cat(Z1_last_lsr)
    Z1_last_lsr = Z1_last_lsr.cpu().numpy()

    W = {}
    resid = {}

    for key, value in Z0.items():
        value = torch.cat(value)
        value = value.cpu().numpy()
        W[key], resid[key] = leastSquares(value, Z1_last_lsr)

    # Augmenting Models
    aug = {}

    for i in indices: 
        model = Augmented(copy.deepcopy(refer), W=W[i], transform_stage=i)
        model = model.eval().to(device)
        aug[i] = model

    # Evaluating
    correct_base = 0
    correct_fine_tuned = 0
    total_samples = 0

    correct = {}
    sim_cls = {}
    for i in indices:
        correct[i] = 0
        sim_cls[i] = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            images = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            total_samples += labels.size(0)

            # Base Model
            logits_base, manipulated_raw_base = base(images)
            preds = logits_base.argmax(dim=1)
            correct_base += (preds == labels).sum().item()

            # Fine-Tuned Model
            logits_fine_tuned, manipulated_raw_fine_tuned = fine_tuned(images)
            preds = logits_fine_tuned.argmax(dim=1)
            correct_fine_tuned += (preds == labels).sum().item()

            # Augmented Base Models
            for i in indices:
                logits_augmented, manipulated_raw_augmented = aug[i](images)
                preds = logits_augmented.argmax(dim=1)
                correct[i] += (preds == labels).sum().item()
                sim_cls[i].append(cosineSimilarity(manipulated_raw_fine_tuned, manipulated_raw_augmented))

    correct_base = correct_base / total_samples
    correct_fine_tuned = correct_fine_tuned / total_samples
    
    for i in indices:
        correct[i] = correct[i] / total_samples
        sim_cls[i] = np.mean(sim_cls[i])
    print(f"Augmented ResNet50 on Entire Cifar-10 Results")
    for i in indices:
        print(f"\tAugmented {i+46} - Last (49) Layer Accuracy: {correct[i]}")
        print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i+1} Layer: {sim_cls[i]:.4f}")
    print(f"Base Accuracy: {correct_base:.4f}")
    print(f"Fine-Tuned Accuracy: {correct_fine_tuned:.4f}")

    # Saving Results
    folder = f"./Cifar10/Entire_Transformation_Matrix_W"
    os.makedirs(folder, exist_ok=True)

    data = {
        'Train_Data_Size': [train_size]*len(indices),
        "Transformation": ["Transformation_Matrix_W"] * len(indices),
        'W': [W[i] for i in indices], # data.loc[0, "W"] -> First layer transformation
        'Residuals': [resid[i] for i in indices],
        'Accuracy': [correct[i] for i in indices], 
        "Co_Sim_CLS": [sim_cls[i] for i in indices],
    }

    df = pd.DataFrame(data, index=[i +46 for i in indices])

    name = f"Transformation_Matrix_W_Entire_Size_{train_size}_Augmentation_Results.csv"
    path = os.path.join(folder, name)
    df.to_csv(path)


print("Testing Ablation Results")

# Augmenting Models
aug = {}

for i in indices: 
    model = Augmented(copy.deepcopy(refer), classifier=f_t.classifier, transform_stage=i)
    model = model.eval().to(device)
    aug[i] = model

# Evaluating
correct_base = 0
correct_fine_tuned = 0
total_samples = 0

correct = {}
sim_cls = {}
for i in indices:
    correct[i] = 0
    sim_cls[i] = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        images = batch["pixel_values"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        total_samples += labels.size(0)

        # Base Model
        logits_base, manipulated_raw_base = base(images)
        preds = logits_base.argmax(dim=1)
        correct_base += (preds == labels).sum().item()

        # Fine-Tuned Model
        logits_fine_tuned, manipulated_raw_fine_tuned = fine_tuned(images)
        preds = logits_fine_tuned.argmax(dim=1)
        correct_fine_tuned += (preds == labels).sum().item()

        # Augmented Base Models
        for i in indices:
            logits_augmented, manipulated_raw_augmented = aug[i](images)
            preds = logits_augmented.argmax(dim=1)
            correct[i] += (preds == labels).sum().item()
            sim_cls[i].append(cosineSimilarity(manipulated_raw_fine_tuned, manipulated_raw_augmented))

for i in indices:
    correct[i] = correct[i] / total_samples
    sim_cls[i] = np.mean(sim_cls[i])
print(f"Augmented Ablation ResNet50 Results")
for i in indices:
    print(f"\tAugmented {i+1} - Last (49) Layer Accuracy: {correct[i]}")
    print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i+1} Layer: {sim_cls[i]:.4f}")
print(f"Base Accuracy: {correct_base:.4f}")
print(f"Fine-Tuned Accuracy: {correct_fine_tuned:.4f}")

# Saving Results
folder = f"./Cifar10/Entire_Transformation_Matrix_W"
os.makedirs(folder, exist_ok=True)

data = {
    'Accuracy': [correct[i] for i in indices], 
    "Co_Sim_CLS": [sim_cls[i] for i in indices],
}

df = pd.DataFrame(data, index=indices)

name = f"Ablation_Results.csv"
path = os.path.join(folder, name)
df.to_csv(path)